# Behind the scenes: Day One
### -> How the data was downloaded

In [ ]:
# imports required for this notebook

import requests
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from workshop_utils import RAW_DIR, PROCESSED_DIR
from workshop_utils.download import (
    download_best, download_gistemp, download_mauna_loa_co2, download_enso,
    download_aod, download_tsi, download_epica, download_vostok_co2,
    download_beck_climate_and_kg,
)

### Berkeley Earth Surface Temperature (BEST) Dataset

In [ ]:
best_yearly = download_best()
#best_yearly.to_csv(PROCESSED_DIR / "day1" / "best_yearly.csv", index=False)
print(f"Saved best_yearly.csv ({len(best_yearly)} rows)")
best_yearly.head()

### NASA GISTEMP Surface Temperature Dataset

In [ ]:
gistemp_yearly = download_gistemp()
#gistemp_yearly.to_csv(PROCESSED_DIR / "day1" / "gistemp_yearly.csv", index=False)
print(f"Saved gistemp_yearly.csv ({len(gistemp_yearly)} rows)")
gistemp_yearly.head()

### Philippines temperature and precipitation
Here, we use ERA5 data to specifically zoom into the Philippines and Bacolod. Below, ERA5 reanalysis data is downloaded for the entire region year by year, with a temperature recording 4 times per day (00, 06, 12, and 18 UTC), and daily precipitation sums.

In [ ]:
# actually, let me go ahead and download from Copernicus API
# bounding box for the Philippines (roughly):
N_lat, S_lat, W_lon, E_lon = 21.25, 4.5, 114, 126.75

from workshop_utils.download import download_era5_copernicus

# download in single year increments
for year in range(1940, 2025):
    download_era5_copernicus(
        variable=["2m_temperature"],
        year=[str(year)],
        month=[f"{i:02d}" for i in range(1, 13)],
        day=[f"{i:02d}" for i in range(1, 32)],
        time=[f"{i:02d}:00" for i in range(0, 24, 6)],
        area=[N_lat, W_lon, S_lat, E_lon],
        output_path=RAW_DIR / "ERA5" / "2m_temperature" / f"t2m_philippines_{year}.nc",
    )

### ERA5 climatology for the Philippines (1991–2020)
Compute the 30-year mean temperature and annual mean precipitation from the downloaded raw files and save as NetCDF for use in the student notebooks.

In [ ]:
# Compute 1991–2020 climatology means and save as NetCDF
t2m_clim_files = sorted(RAW_DIR.glob("ERA5/2m_temperature/t2m_philippines_*.nc"))
t2m_clim_files = [f for f in t2m_clim_files if 1991 <= int(f.stem.split("_")[-1]) <= 2020]

ds_t2m_clim = xr.open_mfdataset(t2m_clim_files, combine="by_coords", drop_variables=["number"])
t2m_mean = (ds_t2m_clim["t2m"].mean("valid_time") - 273.15).compute()

tp_clim_files = sorted(RAW_DIR.glob("ERA5/precipitation/tp_philippines_*.nc"))
tp_clim_files = [f for f in tp_clim_files if 1991 <= int(f.stem.split("_")[-1]) <= 2020]

ds_tp_clim = xr.open_mfdataset(tp_clim_files, combine="by_coords", drop_variables=["number"])
tp_mean_annual = (ds_tp_clim["tp"].mean("valid_time") * 1000 * 365).compute()

# uncomment to save:
#t2m_mean.to_netcdf(PROCESSED_DIR / "day1" / "t2m_clim_mean.nc")
#tp_mean_annual.to_netcdf(PROCESSED_DIR / "day1" / "tp_clim_annual_mean.nc")
print("Saved t2m_clim_mean.nc and tp_clim_annual_mean.nc")

### Bacolod ERA5 time series (1940–2024)
Extract the Bacolod grid point from the full ERA5 record and compute the yearly temperature series and monthly climatologies (1991–2020) used in Part III.

In [ ]:
BACOLOD_LAT = 10.68
BACOLOD_LON = 122.95

all_t2m_files = sorted(RAW_DIR.glob("ERA5/2m_temperature/t2m_philippines_*.nc"))
ds_t2m_all = xr.open_mfdataset(all_t2m_files, combine="by_coords", drop_variables=["number"])
t2m_bacolod = (ds_t2m_all["t2m"]
               .sel(latitude=BACOLOD_LAT, longitude=BACOLOD_LON, method="nearest")
               - 273.15)

t2m_yearly  = t2m_bacolod.groupby("valid_time.year").mean().compute()
t2m_monthly = (t2m_bacolod
               .sel(valid_time=slice("1991", "2020"))
               .groupby("valid_time.month").mean().compute())

all_tp_files = sorted(RAW_DIR.glob("ERA5/precipitation/tp_philippines_*.nc"))
ds_tp_all = xr.open_mfdataset(all_tp_files, combine="by_coords", drop_variables=["number"])
tp_bacolod = (ds_tp_all["tp"]
              .sel(latitude=BACOLOD_LAT, longitude=BACOLOD_LON, method="nearest")
              * 1000)  # m/day → mm/day
tp_monthly = (tp_bacolod
              .sel(valid_time=slice("1991", "2020"))
              .resample(valid_time="MS").sum()           # sum daily values → monthly total (mm/month)
              .groupby("valid_time.month").mean()        # average each calendar month across 1991–2020
              .compute())

# uncomment to save:
# t2m_yearly.to_netcdf(PROCESSED_DIR / "day1" / "t2m_bacolod_yearly.nc")   # day1-only
# t2m_monthly.to_netcdf(PROCESSED_DIR / "t2m_bacolod_monthly.nc")         # shared with Day 2 -- stays top-level
#tp_monthly.to_netcdf(PROCESSED_DIR / "tp_bacolod_monthly.nc")            # shared with Day 2 -- stays top-level
print("Saved t2m_bacolod_yearly.nc, t2m_bacolod_monthly.nc, tp_bacolod_monthly.nc")

### CHIRPS Precipitation Dataset

Might not use this, actually. Let's keep it in for now.

In [ ]:
url = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/netcdf/chirps-v2.0.monthly.nc"
ds = xr.open_dataset(url)
# ...

### Mauna Loa CO2 record -- NOAA Global Monitoring Laboratory

In [ ]:
co2_monthly, co2_yearly = download_mauna_loa_co2()
#co2_monthly.to_csv(PROCESSED_DIR / "day1" / "co2_monthly.csv", index=False)
print(f"Saved co2_monthly.csv ({len(co2_monthly)} rows)")
#co2_yearly.to_csv(PROCESSED_DIR / "day1" / "co2_yearly.csv", index=False)
print(f"Saved co2_yearly.csv ({len(co2_yearly)} rows)")

### ENSO (Niño3.4 Index) -- NOAA Physical Sciences Laboratory via KNMI

In [ ]:
nino_yearly = download_enso()
#nino_yearly.to_csv(PROCESSED_DIR / "day1" / "enso_yearly.csv", index=False)
print(f"Saved enso_yearly.csv ({len(nino_yearly)} rows)")

### Aerosol Optical Depth (AOD, proxy for volcanic activity) -- NASA GISS
Should appear as large spikes:
- 1963  Agung
- 1982  El Chichón
- 1991  Pinatubo

In [ ]:
aod_df = download_aod()
#aod_df.to_csv(PROCESSED_DIR / "day1" / "aod_global.csv", index=False)
print(f"Saved aod_global.csv ({len(aod_df)} rows)")

### Total Solar Irradiance (TSI) -- NOAA

In [ ]:
tsi_monthly, tsi_yearly = download_tsi()
#tsi_monthly.to_csv(PROCESSED_DIR / "day1" / "tsi_monthly.csv", index=False)
print(f"Saved tsi_monthly.csv ({len(tsi_monthly)} rows)")
#tsi_yearly.to_csv(PROCESSED_DIR / "day1" / "tsi_yearly.csv", index=False)
print(f"Saved tsi_yearly.csv ({len(tsi_yearly)} rows)")

### EPICA Dome C Ice Core

In [ ]:
epica = download_epica()
#epica.to_csv(PROCESSED_DIR / "day1" / "epica.csv", index=False)
print(f"Saved epica.csv ({len(epica)} rows)")

In [ ]:
vostok = download_vostok_co2()
#vostok.to_csv(PROCESSED_DIR / "day1" / "vostok_co2.csv", index=False)
print(f"Saved vostok_co2.csv ({len(vostok)} rows)")

In [ ]:
# Vostok CO₂ + Mauna Loa extension — 800,000-year CO₂ context
mauna_loa = co2_monthly.copy()
mauna_loa["bp"] = 1950 - mauna_loa["decimal date"]

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(vostok["age_ka"] * 1000, vostok["co2_ppm"],
        color="steelblue", linewidth=1.4, label="EPICA Dome C / Vostok")
ax.plot(mauna_loa["bp"], mauna_loa["average"],
        color="darkorange", linewidth=1.8, label="Mauna Loa")

ax.axvline(0, color="gray", linewidth=0.9, linestyle="--")
ax.text(-2000, 190, "1950 CE", color="gray", fontsize=9, rotation=90, va="bottom")

vostok_max = vostok.loc[vostok["co2_ppm"].idxmax()]
mauna_max  = mauna_loa.loc[mauna_loa["average"].idxmax()]

ax.scatter(vostok_max["age_ka"] * 1000, vostok_max["co2_ppm"],
           s=60, color="steelblue", edgecolor="black", zorder=5)
ax.scatter(mauna_max["bp"], mauna_max["average"],
           s=60, color="darkorange", edgecolor="black", zorder=5)

ax.annotate(f'Highest previous: {vostok_max["co2_ppm"]:.0f} ppm',
            xy=(vostok_max["age_ka"] * 1000, vostok_max["co2_ppm"]),
            xytext=(30, 15), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", lw=0.8))
ax.annotate(f'Today: {mauna_max["average"]:.1f} ppm',
            xy=(mauna_max["bp"], mauna_max["average"]),
            xytext=(-180, -20), textcoords="offset points",
            arrowprops=dict(arrowstyle="->", lw=0.8))

ax.set_xlim(450000, -10000)
ax.set_xlabel("Years before present (BP; present = 1950 CE)")
ax.set_ylabel("Atmospheric CO₂ (ppm)")
ax.set_title("Atmospheric CO₂ over 800,000 years")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Köppen-Geiger climate maps from Beck et al.

In [ ]:
beck = download_beck_climate_and_kg()

# All five files are shared reference layers reused across Day 1/2/3 -- stay at PROCESSED_DIR top level
for out_name, ds_ph in beck.items():
    out_path = PROCESSED_DIR / out_name
    #ds_ph.to_netcdf(out_path)
    print(f"{out_name}  {dict(ds_ph.sizes)}")

### Land–sea mask at 0.1° (matches the Beck climate ensemble grid)

Beck et al.'s land climate ensemble has no value over open water — every grid cell is `NaN` in all 12 months if it's ocean. That NaN pattern *is* a land–sea mask, for free. We save it out explicitly (1 = land, 0 = ocean) at the same 170×135 / 0.1° grid as `climate_philippines_*.nc`, for use in Day 2 (Part I map-making). Note this is a different grid than `land_sea_mask.nc` (ERA5's own 0.25° grid, used in Day 1).

In [ ]:
# Derive land/sea from the NaN pattern of the present-day (1991-2020) climate ensemble
clim_present = xr.open_dataset(PROCESSED_DIR / "climate_philippines_1991_2020_mean.nc")
is_land = clim_present["air_temperature"].notnull().all("time")

land_sea_mask_0p1 = is_land.astype("float32").rename("land_sea_mask")
land_sea_mask_0p1.attrs["description"] = "1 = land, 0 = ocean; derived from NaN pattern in Beck et al. climate ensemble (0.1 deg)"

# uncomment to save:
#land_sea_mask_0p1.to_netcdf(PROCESSED_DIR / "land_sea_mask_0p1.nc")
print(f"Saved land_sea_mask_0p1.nc  {dict(land_sea_mask_0p1.sizes)}  land pixels: {int(is_land.sum())}/{is_land.size}")

### Land–sea mask at 0.00833333° (~1 km, matches the Beck KG classification grid)

`kg_philippines_present.nc`/`kg_philippines_future_ssp460.nc` use `0` as a distinct "ocean / no-data" sentinel value (not `NaN`) — confirmed by count: exactly 2,864,741 of 3,304,800 pixels are `0` in both the present and future files, matching the ocean fraction and staying identical across time periods (same land geography). Same free-mask trick as the 0.1° version above, just derived from a different file/resolution — used to properly mask ocean out of the Beck KG maps (see the plotting fix in `notebooks/day1.ipynb`, Part I).

In [ ]:
# Derive land/sea from the 0 sentinel in the present-day KG classification (1 km grid)
ds_kg_present = xr.open_dataset(PROCESSED_DIR / "kg_philippines_present.nc")
is_land_1km = ds_kg_present["kg_class"] > 0

land_sea_mask_0p00833333 = is_land_1km.astype("float32").rename("land_sea_mask")
land_sea_mask_0p00833333.attrs["description"] = "1 = land, 0 = ocean/no-data; derived from the 0 sentinel in Beck et al. koppen_geiger_0p00833333.nc (kg_philippines_present.nc), ~1km resolution"

# uncomment to save:
#land_sea_mask_0p00833333.to_netcdf(PROCESSED_DIR / "land-sea-mask_0p00833333.nc")
print(f"Saved land-sea-mask_0p00833333.nc  {dict(land_sea_mask_0p00833333.sizes)}  land pixels: {int(is_land_1km.sum())}/{is_land_1km.size}")